# ARCH 6133 Places / Platforms  
## Points of Interest Data: Platform Definitions of Place

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danmillr/places-platforms/blob/main/tutorials/ARCH6133_POI_Data.ipynb)

---


## Documentation links

Keep these open as you work:

### Google Places API
- Overview: https://developers.google.com/maps/documentation/places/web-service/overview
- Nearby Search: https://developers.google.com/maps/documentation/places/web-service/nearby-search
- Text Search: https://developers.google.com/maps/documentation/places/web-service/text-search
- Place Details: https://developers.google.com/maps/documentation/places/web-service/place-details
- Place Data Fields: https://developers.google.com/maps/documentation/places/web-service/data-fields
- Place Types: https://developers.google.com/maps/documentation/places/web-service/place-types
- Popular times support note: https://support.google.com/business/answer/6263531?hl=en

### Foursquare
- Place Search: https://docs.foursquare.com/developer/reference/place-search
- Response Fields: https://docs.foursquare.com/developer/reference/response-fields
- Places Pro and Premium schema, including `hours_popular`: https://docs.foursquare.com/data-products/docs/places-pro-and-premium

### OpenStreetMap
- Map features: https://wiki.openstreetmap.org/wiki/Map_features
- `amenity=*`: https://wiki.openstreetmap.org/wiki/Key:amenity
- Overpass API by example: https://wiki.openstreetmap.org/wiki/Overpass_API/Overpass_API_by_Example
- Overpass Turbo: https://overpass-turbo.eu/
- OSMnx features module: https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx-features-module

### Overture Maps
- Places Guide: https://docs.overturemaps.org/guides/places/
- Quickstart: https://docs.overturemaps.org/getting-data/
- DuckDB guide: https://docs.overturemaps.org/getting-data/duckdb/
- Place schema: https://docs.overturemaps.org/schema/reference/places/place/


## Conceptual setup: four models of place

| Source | What it thinks a place is | What it is especially useful for | What to watch critically |
|---|---|---|---|
| Google Places | A searchable and navigable destination connected to Google Maps and Search | names, addresses, hours, ratings, reviews, categories, accessibility, amenities, map links | the API exposes only a controlled subset of platform knowledge |
| Foursquare Places | A venue or POI enriched by categories, visits, check-ins, popularity, and location intelligence | popularity, popular hours, venue metadata, chains, rich categories | behavioral proxies depend on tracking infrastructures and sufficient activity |
| OpenStreetMap | A volunteered geographic feature encoded through tags | civic and infrastructural detail, open data, editability, non-commercial features | coverage and tagging are uneven and community-shaped |
| Overture Maps | An open, distributed point representation of real-world entities compiled from multiple sources | bulk data, confidence, source attribution, GeoParquet, GERS-aware IDs | source conflation and standardization require interpretation |

**Important Google caveat:** Google Maps may show popular times, live busyness, wait times, and typical visit duration for some businesses, but these are not standard public fields in the current Places API field list. Treat this as a productive gap: the API is not the platform.


## Data pipeline for today

```mermaid
flowchart LR
    A[Choose study area] --> B[Query platform sources]
    B --> C[Save raw JSON or GeoJSON]
    C --> D[Flatten source-specific fields]
    D --> E[Normalize selected fields]
    E --> F[Compare coverage and metadata]
    F --> G[Map and visualize]
    G --> H[Reflect on absences and prototype uses]
```

If your notebook environment does not render Mermaid diagrams, read the code block as a text diagram.


# 00. Setup

Run this section first. In Google Colab, uncomment the install line. If you are running locally and already have the packages installed, you can skip it.


In [ ]:
# In Google Colab, uncomment this line if needed.
# !pip -q install pandas geopandas shapely requests folium matplotlib tqdm osmnx overturemaps duckdb ipywidgets

from pathlib import Path
import json
import os
import time
import math
import getpass
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
import folium
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

DATA_DIR = Path('poi_data')
RAW_DIR = DATA_DIR / 'raw'
OUT_DIR = DATA_DIR / 'output'
for d in [DATA_DIR, RAW_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Data folders ready:', DATA_DIR.resolve())


## API keys and access for this notebook

> **Run this section before the API tutorials.**  
> This notebook uses several different place-data sources. Some require API keys and some do not.

| Source | API key needed? | When you need it | Notes |
|---|---:|---|---|
| **Google Places API** | Yes | Google Nearby Search and Place Details cells | Requires a Google Maps Platform project, billing enabled, and the Places API enabled. Keep field masks modest to control cost. |
| **Foursquare Places API** | Yes | Foursquare Search and rich metadata cells | Some richer fields may depend on your Foursquare account, product tier, or available response fields. |
| **OpenStreetMap / Overpass** | No | OSM POI queries | Public service. Keep bounding boxes small and avoid repeated large requests. |
| **Overture Maps Places** | No | Overture CLI or DuckDB GeoParquet examples | Public open dataset. Requires internet access and package installation, but not an API key. |

### Recommended key storage

Do **not** hard-code API keys into a notebook that will be uploaded to GitHub or shared with students.

In Colab, use the **Secrets** panel in the left sidebar and add:

```text
GOOGLE_API_KEY
FOURSQUARE_API_KEY
```

Locally, you can set environment variables with the same names. As a fallback, the setup cell below will ask you to paste a key at runtime. Press Enter to skip a provider you do not plan to use.


In [ ]:
# --- API KEY SETUP ------------------------------------------------------------
# Run this once near the beginning of the notebook.
#
# This cell defines all external API keys used later in the tutorial.
# Keys are read in this order:
#   1. Environment variables, such as GOOGLE_API_KEY
#   2. Google Colab Secrets, if running in Colab
#   3. Manual prompt using getpass, so the key is not displayed in output
#
# Sources that do NOT need keys:
#   - OpenStreetMap / Overpass
#   - Overture Maps public GeoParquet data

import os
import getpass

# Optional: read from Google Colab Secrets, if available.
try:
    from google.colab import userdata  # type: ignore
except Exception:
    userdata = None


def read_secret(name: str, aliases=None):
    """
    Read a secret from environment variables or Colab Secrets.

    Parameters
    ----------
    name : str
        Primary secret name.
    aliases : list[str] | None
        Alternative names to check.

    Returns
    -------
    str | None
        The secret value, or None if not found.
    """
    candidates = [name] + (aliases or [])

    for key_name in candidates:
        value = os.environ.get(key_name)
        if value:
            return value

    if userdata is not None:
        for key_name in candidates:
            try:
                value = userdata.get(key_name)
                if value:
                    return value
            except Exception:
                pass

    return None


def prompt_for_key(label: str, existing_value=None):
    """
    Prompt for a key only if it was not already found.
    Press Enter to skip that provider.
    """
    if existing_value:
        print(f"{label}: found")
        return existing_value

    value = getpass.getpass(f"{label}: paste key or press Enter to skip: ").strip()
    if value:
        print(f"{label}: added for this runtime")
        return value

    print(f"{label}: skipped")
    return None


# Required only for the Google Places section.
GOOGLE_API_KEY = read_secret(
    "GOOGLE_API_KEY",
    aliases=["GOOGLE_PLACES_API_KEY", "GOOGLE_MAPS_API_KEY"]
)
GOOGLE_API_KEY = prompt_for_key("Google Places API key", GOOGLE_API_KEY)

# Required only for the Foursquare section.
FOURSQUARE_API_KEY = read_secret(
    "FOURSQUARE_API_KEY",
    aliases=["FSQ_API_KEY", "FOURSQUARE_PLACES_API_KEY"]
)
FOURSQUARE_API_KEY = prompt_for_key("Foursquare API key", FOURSQUARE_API_KEY)

# Convenience flags used later in the notebook.
HAS_GOOGLE_KEY = bool(GOOGLE_API_KEY)
HAS_FOURSQUARE_KEY = bool(FOURSQUARE_API_KEY)

print("\nKey status")
print("Google Places:", "ready" if HAS_GOOGLE_KEY else "not configured")
print("Foursquare:", "ready" if HAS_FOURSQUARE_KEY else "not configured")
print("OpenStreetMap / Overpass: no key required")
print("Overture Maps Places: no key required")


# 01. Define a study area

You can describe the study area in **either** of two ways. The cell below uses the same downstream variables (`BBOX`, `CENTER_LAT`, `CENTER_LON`, `RADIUS_METERS`, and an optional `STUDY_POLYGON`) either way.

**Option A — place name (recommended for class).** Type a neighborhood, district, or city name. The notebook geocodes it through OpenStreetMap's Nominatim service via OSMnx and uses the returned polygon. Be specific so Nominatim returns the right shape, for example:

```text
East Harlem, Manhattan, New York City, New York, USA
Washington Heights, Manhattan, New York City, New York, USA
Greenpoint, Brooklyn, New York City, New York, USA
Ithaca, New York, USA
Manhattan, New York City, New York, USA   <-- large; APIs will cap results
```

**Option B — bounding box.** Supply coordinates directly, in the order:

```text
min_lon, min_lat, max_lon, max_lat
```

A center point and a circumscribing radius are derived automatically for APIs that prefer circular searches.

**Scale matters.** Google Nearby Search returns at most 20 results per type per call, Foursquare at most 50 per query, and Overpass and Overture will happily return everything in your bounding box. For a clean cross-source comparison, pick a 1–5 km² neighborhood. Larger areas (a full borough, a city) will silently truncate Google and Foursquare results and will skew the comparison.


In [ ]:
# --- STUDY AREA CONTROLS -----------------------------------------------------
# Change any value below. When you're happy with the settings, run the next cell
# to apply them and preview the area on a map.

study_area_mode = widgets.ToggleButtons(
    options=[('Place name (geocode)', 'name'), ('Bounding box', 'bbox')],
    value='name',
    description='Mode:',
    style={'description_width': '110px'},
)

place_name_widget = widgets.Text(
    value='East Harlem, Manhattan, New York, USA',
    description='Place name:',
    placeholder='e.g. Greenpoint, Brooklyn, New York, USA',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '110px'},
)

fallback_radius_widget = widgets.IntSlider(
    value=900, min=200, max=3000, step=100,
    description='Fallback radius (m):',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '160px'},
)
fallback_radius_help = widgets.HTML(
    "<div style='font-size:11px;color:#666;margin-left:165px;'>"
    "Used when OSM has no boundary polygon for the place (common for small NYC neighborhoods)."
    "</div>"
)

bbox_widget = widgets.Text(
    value='-73.955, 40.795, -73.930, 40.815',
    description='BBOX:',
    placeholder='min_lon, min_lat, max_lon, max_lat',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '110px'},
)
bbox_help = widgets.HTML(
    "<div style='font-size:11px;color:#666;margin-left:115px;'>"
    "Order: min_lon, min_lat, max_lon, max_lat. Used only in Bounding box mode."
    "</div>"
)

display(widgets.VBox([
    widgets.HTML("<b>Study area</b>"),
    study_area_mode,
    place_name_widget,
    fallback_radius_widget,
    fallback_radius_help,
    bbox_widget,
    bbox_help,
]))


In [ ]:
# --- APPLY STUDY AREA SETTINGS -----------------------------------------------
# Reads the widgets in the cell above. Re-run this whenever you change them.
#
# Produces:
#   BBOX           (min_lon, min_lat, max_lon, max_lat)
#   CENTER_LAT, CENTER_LON
#   RADIUS_METERS  (circumscribes the bbox; used by APIs that take ll + radius)
#   STUDY_POLYGON  (shapely geometry if Nominatim returned a real polygon, else None)


def _bbox_from_point(lat, lon, radius_m):
    """Build a square bbox of side ~ 2*radius_m around (lat, lon)."""
    dlat = radius_m / 111000.0
    dlon = radius_m / (111000.0 * math.cos(math.radians(lat)))
    return (lon - dlon, lat - dlat, lon + dlon, lat + dlat)


def derive_study_area(place_name=None, bbox=None, fallback_radius_m=900):
    """Return (name, bbox, center_lat, center_lon, radius_m, polygon)."""
    polygon = None
    if place_name:
        try:
            import osmnx as ox
        except ImportError as e:
            raise ImportError(
                "osmnx is required for place-name mode. In Colab, run the install "
                "cell at the top of the notebook (uncomment the !pip line)."
            ) from e
        try:
            gdf = ox.geocode_to_gdf(place_name)
            polygon = gdf.geometry.iloc[0]
            minx, miny, maxx, maxy = polygon.bounds
            print(f"Geocoded '{place_name}' to a polygon.")
        except (TypeError, ValueError):
            lat, lon = ox.geocode(place_name)
            minx, miny, maxx, maxy = _bbox_from_point(lat, lon, fallback_radius_m)
            print(
                f"Geocoded '{place_name}' to a point (no polygon in OSM); "
                f"using a {fallback_radius_m} m square around it."
            )
        name = place_name
    else:
        minx, miny, maxx, maxy = bbox
        name = f'Custom bbox {bbox}'

    center_lat = (miny + maxy) / 2
    center_lon = (minx + maxx) / 2

    R = 6371000.0
    dlat = math.radians(maxy - miny)
    dlon = math.radians(maxx - minx) * math.cos(math.radians(center_lat))
    radius_m = int(R * math.hypot(dlat, dlon) / 2)

    return name, (minx, miny, maxx, maxy), center_lat, center_lon, radius_m, polygon


_use_place_name = (study_area_mode.value == 'name')
_bbox_input = None
if not _use_place_name:
    try:
        _bbox_input = tuple(float(p.strip()) for p in bbox_widget.value.split(','))
        assert len(_bbox_input) == 4
    except (ValueError, AssertionError):
        raise ValueError(
            f"Could not parse BBOX widget value: {bbox_widget.value!r}. "
            "Expected: min_lon, min_lat, max_lon, max_lat"
        )

(STUDY_AREA_NAME, BBOX, CENTER_LAT, CENTER_LON,
 RADIUS_METERS, STUDY_POLYGON) = derive_study_area(
    place_name=place_name_widget.value.strip() if _use_place_name else None,
    bbox=_bbox_input,
    fallback_radius_m=fallback_radius_widget.value,
)

bbox_width_km = (BBOX[2] - BBOX[0]) * 111 * math.cos(math.radians(CENTER_LAT))
bbox_height_km = (BBOX[3] - BBOX[1]) * 111
area_km2 = abs(bbox_width_km * bbox_height_km)

print('Study area:', STUDY_AREA_NAME)
print('BBOX:', BBOX)
print('Center:', round(CENTER_LAT, 5), round(CENTER_LON, 5))
print('Radius (m):', RADIUS_METERS)
print(f'Approx bbox area: {area_km2:.1f} km^2')

if area_km2 > 25:
    print()
    print('WARNING: large study area.')
    print('  Google Nearby Search caps at 20 results per type per call.')
    print('  Foursquare caps at 50 results per query.')
    print('  Results from those two sources will be a small, biased sample.')
    print('  For a clean comparison, pick a 1-5 km^2 neighborhood.')

zoom = 13 if area_km2 > 10 else 15
m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=zoom, tiles='CartoDB positron')
folium.Rectangle(
    bounds=[[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]],
    color='#888', weight=1, fill=False, dash_array='4 4'
).add_to(m)
folium.Circle(
    location=[CENTER_LAT, CENTER_LON], radius=RADIUS_METERS,
    color='#888', weight=1, fill=False
).add_to(m)
if STUDY_POLYGON is not None:
    folium.GeoJson(
        STUDY_POLYGON.__geo_interface__,
        style_function=lambda f: {'color': '#3C4ED6', 'weight': 2, 'fillOpacity': 0.08}
    ).add_to(m)
m


## Helper functions

These functions save files, flatten nested fields, convert rows to GeoJSON, and make quick maps and charts.


In [ ]:
from folium.plugins import MarkerCluster


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print('Saved', path)


def load_json(path: Path) -> Any:
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def safe_get(d: Dict, keys: List[str], default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def rows_to_geojson(rows: List[Dict], lon_col='lon', lat_col='lat') -> Dict:
    features = []
    for row in rows:
        lon, lat = row.get(lon_col), row.get(lat_col)
        if lon is None or lat is None:
            continue
        props = {k: v for k, v in row.items() if k not in [lon_col, lat_col]}
        features.append({
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [float(lon), float(lat)]},
            'properties': props
        })
    return {'type': 'FeatureCollection', 'features': features}


def save_geojson(rows: List[Dict], path: Path, lon_col='lon', lat_col='lat') -> Dict:
    gj = rows_to_geojson(rows, lon_col=lon_col, lat_col=lat_col)
    save_json(gj, path)
    return gj


def plot_counts(df: pd.DataFrame, column: str, title: str, top_n=15):
    counts = df[column].fillna('missing').astype(str).value_counts().head(top_n).sort_values()
    ax = counts.plot(kind='barh', figsize=(8, max(3, 0.35 * len(counts))))
    ax.set_title(title)
    ax.set_xlabel('Count')
    ax.set_ylabel(column)
    plt.tight_layout()
    plt.show()


def first_nonempty(*vals):
    for v in vals:
        if v not in [None, '', [], {}] and not (isinstance(v, float) and math.isnan(v)):
            return v
    return None


# --- Interactive map helpers used by every source section --------------------

CATEGORY_PALETTE = [
    '#3C4ED6', '#1B1B33', '#D6873C', '#3CB371', '#A23CD6',
    '#D63C5E', '#3CB6D6', '#D6C13C', '#7A3CD6', '#5E8B3C',
]

# Stable colors for the four sources so the combined map is readable.
SOURCE_COLORS = {
    'google_places': '#3C4ED6',
    'foursquare': '#D6873C',
    'openstreetmap': '#3CB371',
    'overture': '#A23CD6',
}


def color_for(value, lookup):
    """Stable color per category, assigned in first-seen order."""
    if value not in lookup:
        lookup[value] = CATEGORY_PALETTE[len(lookup) % len(CATEGORY_PALETTE)]
    return lookup[value]


def _build_popup(row, name_col, category_col):
    name = row.get(name_col) or 'Unnamed'
    parts = [f'<div style="font-weight:600;font-size:13px;">{name}</div>']
    src = row.get('source')
    if src:
        parts.append(f'<div style="font-size:10px;color:#888;text-transform:uppercase;letter-spacing:0.04em;">{src}</div>')
    cat = row.get(category_col)
    if cat:
        parts.append(f'<div style="font-size:11px;color:#555;margin-top:2px;">{cat}</div>')
    if row.get('address'):
        parts.append(f'<div style="font-size:11px;margin-top:4px;">{row["address"]}</div>')
    facts = []
    if pd.notna(row.get('rating')):
        rc = row.get('review_count')
        rc_str = f' ({int(rc):,} reviews)' if pd.notna(rc) else ''
        facts.append(f'★ {row["rating"]}{rc_str}')
    if row.get('price'):
        facts.append(f'price: {row["price"]}')
    if pd.notna(row.get('popularity')):
        facts.append(f'popularity: {row["popularity"]:.2f}')
    if pd.notna(row.get('confidence')):
        facts.append(f'confidence: {row["confidence"]:.2f}')
    for status_field in ['business_status', 'operating_status', 'closed_bucket']:
        v = row.get(status_field)
        if v and str(v).lower() not in ('operational', 'open', 'nan'):
            facts.append(str(v))
    if facts:
        parts.append(f'<div style="font-size:11px;margin-top:4px;">{" · ".join(facts)}</div>')
    links = []
    if row.get('website'):
        links.append(f'<a href="{row["website"]}" target="_blank">website</a>')
    if row.get('maps_url'):
        links.append(f'<a href="{row["maps_url"]}" target="_blank">Google Maps</a>')
    if links:
        parts.append(f'<div style="font-size:11px;margin-top:6px;">{" · ".join(links)}</div>')
    return ''.join(parts)


def exploratory_map(
    df: pd.DataFrame,
    name_col: str = 'name',
    category_col: str = 'category_original',
    color_by: Optional[str] = None,
    lat_col: str = 'lat',
    lon_col: str = 'lon',
    cluster: bool = True,
    title: Optional[str] = None,
    max_legend_rows: int = 20,
):
    """Folium map with marker clustering, color-by group, rich popups, layer toggle, and legend.

    color_by defaults to category_col. Pass color_by='source' on the combined dataframe
    to color each source distinctly.
    """
    color_by = color_by or category_col
    m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=15, tiles='CartoDB positron')
    folium.Rectangle(
        bounds=[[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]],
        color='#888', weight=1, fill=False, dash_array='4 4'
    ).add_to(m)
    if STUDY_POLYGON is not None:
        folium.GeoJson(
            STUDY_POLYGON.__geo_interface__,
            name='Study area',
            style_function=lambda f: {'color': '#3C4ED6', 'weight': 1, 'fillOpacity': 0.04},
        ).add_to(m)

    if not len(df):
        return m

    plot_df = df.dropna(subset=[lat_col, lon_col]).copy()
    if color_by not in plot_df.columns:
        plot_df[color_by] = 'all'
    plot_df[color_by] = plot_df[color_by].fillna('uncategorized').astype(str)

    # For combined views, use the stable SOURCE_COLORS palette.
    color_lookup = dict(SOURCE_COLORS) if color_by == 'source' else {}
    layers = {}
    for group_value, group in plot_df.groupby(color_by):
        color = color_for(group_value, color_lookup)
        fg = folium.FeatureGroup(name=f'{group_value} ({len(group)})', show=True)
        target = MarkerCluster().add_to(fg) if cluster else fg
        for _, row in group.iterrows():
            popup_html = _build_popup(row, name_col, category_col)
            folium.CircleMarker(
                location=[row[lat_col], row[lon_col]],
                radius=6, color=color, weight=1,
                fill=True, fill_color=color, fill_opacity=0.85,
                tooltip=str(row.get(name_col) or 'Unnamed'),
                popup=folium.Popup(popup_html, max_width=320),
            ).add_to(target)
        fg.add_to(m)
        layers[group_value] = color

    folium.LayerControl(collapsed=False).add_to(m)

    # Inline legend, truncated for very long category lists.
    legend_items = list(layers.items())
    truncated = len(legend_items) > max_legend_rows
    legend_items = legend_items[:max_legend_rows]
    legend_rows = ''.join(
        f'<div style="display:flex;align-items:center;margin:2px 0;">'
        f'<span style="display:inline-block;width:10px;height:10px;background:{c};'
        f'border-radius:50%;margin-right:6px;"></span>'
        f'<span style="font-size:11px;">{g}</span></div>'
        for g, c in legend_items
    )
    if truncated:
        legend_rows += f'<div style="font-size:10px;color:#888;margin-top:4px;">+ more (toggle layers)</div>'
    title_html = f'<div style="font-weight:600;margin-bottom:4px;">{title}</div>' if title else ''
    legend_html = f"""
    <div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999;
                background: white; padding: 10px 12px; border: 1px solid #ccc;
                border-radius: 4px; font-family: Open Sans, sans-serif;
                max-height: 280px; overflow-y: auto;">
        {title_html}{legend_rows}
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    return m

print('Helpers ready')


# 02. Google Places API

Google Places is useful for structured place records tied to Google Maps and Search. We will use Nearby Search and optionally Place Details.

Important details:

- Nearby Search uses a POST request to `https://places.googleapis.com/v1/places:searchNearby`.
- Requests require a field mask through the `X-Goog-FieldMask` header.
- Field masks control cost, latency, and returned fields.
- Do not use `*` for class exercises unless you are intentionally testing and understand the billing implications.
- Popular times, live busyness, wait times, and typical visit duration are described in Google support documentation, but they are not standard public Places API fields in the current field list.

Before running this section, create a Google Maps Platform API key with Places API enabled.


In [ ]:
# GOOGLE_API_KEY is defined in the consolidated API key setup cell above.
GOOGLE_NEARBY_URL = 'https://places.googleapis.com/v1/places:searchNearby'
GOOGLE_DETAILS_URL = 'https://places.googleapis.com/v1/places/{place_id}'

# Keep this field mask modest for class use.
GOOGLE_NEARBY_FIELD_MASK = ','.join([
    'places.id',
    'places.displayName',
    'places.formattedAddress',
    'places.location',
    'places.primaryType',
    'places.types',
    'places.businessStatus',
    'places.rating',
    'places.userRatingCount',
    'places.priceLevel',
    'places.regularOpeningHours',
    'places.currentOpeningHours',
    'places.websiteUri',
    'places.googleMapsUri'
])

print(GOOGLE_NEARBY_FIELD_MASK)

## Google Nearby Search

The `includedTypes` list must use Google place type names. Try one or more of the following:

```text
restaurant, cafe, library, museum, school, hospital, park, supermarket, church, transit_station
```

The search below runs one request per included type, then deduplicates by Google place ID.


In [ ]:
# --- GOOGLE PLACES CONTROLS --------------------------------------------------
# Hold cmd / ctrl to multi-select types. Add any extras (comma-separated) below.
# Google caps Nearby Search at 20 results per type per call.

GOOGLE_TYPE_OPTIONS = [
    'restaurant', 'cafe', 'bar', 'bakery', 'meal_takeaway',
    'park', 'library', 'museum', 'art_gallery', 'tourist_attraction',
    'school', 'university', 'hospital', 'pharmacy', 'doctor',
    'supermarket', 'grocery_or_supermarket', 'shopping_mall', 'store',
    'church', 'mosque', 'synagogue', 'hindu_temple',
    'transit_station', 'subway_station', 'bus_station', 'train_station',
    'bank', 'atm', 'post_office', 'gym',
]

google_types_widget = widgets.SelectMultiple(
    options=GOOGLE_TYPE_OPTIONS,
    value=('restaurant', 'cafe', 'library', 'park'),
    description='Types:',
    rows=10,
    layout=widgets.Layout(width='420px'),
    style={'description_width': '90px'},
)

google_types_extra = widgets.Text(
    value='',
    description='Add extras:',
    placeholder='comma-separated, e.g. movie_theater, night_club',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '90px'},
)

google_max_per_type = widgets.IntSlider(
    value=20, min=1, max=20, step=1,
    description='Max per type:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '110px'},
)
google_max_help = widgets.HTML(
    "<div style='font-size:11px;color:#666;margin-left:115px;'>"
    "Hard cap is 20 per call (Google API). Total calls = number of selected types."
    "</div>"
)

display(widgets.VBox([
    widgets.HTML("<b>Google Places — query controls</b>"),
    google_types_widget,
    google_types_extra,
    google_max_per_type,
    google_max_help,
]))


In [ ]:
# GOOGLE_API_KEY is defined in the consolidated API key setup cell above.
def google_nearby_search(place_type: str, max_result_count: int = 20) -> Dict:
    if not GOOGLE_API_KEY:
        raise ValueError('No Google API key provided')
    payload = {
        'includedTypes': [place_type],
        'maxResultCount': max_result_count,
        'locationRestriction': {
            'circle': {
                'center': {'latitude': CENTER_LAT, 'longitude': CENTER_LON},
                'radius': RADIUS_METERS
            }
        }
    }
    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': GOOGLE_API_KEY,
        'X-Goog-FieldMask': GOOGLE_NEARBY_FIELD_MASK
    }
    r = requests.post(GOOGLE_NEARBY_URL, headers=headers, json=payload, timeout=30)
    if r.status_code != 200:
        print(r.status_code, r.text[:1000])
    r.raise_for_status()
    return r.json()


def flatten_google_place(place: Dict, query_type: Optional[str] = None) -> Dict:
    loc = place.get('location', {})
    display_name = place.get('displayName', {}) or {}
    return {
        'source': 'google_places',
        'source_id': place.get('id'),
        'name': display_name.get('text'),
        'category_original': first_nonempty(place.get('primaryType'), ','.join(place.get('types', [])[:3]) if place.get('types') else None),
        'query_type': query_type,
        'lat': loc.get('latitude'),
        'lon': loc.get('longitude'),
        'address': place.get('formattedAddress'),
        'business_status': place.get('businessStatus'),
        'rating': place.get('rating'),
        'review_count': place.get('userRatingCount'),
        'price': place.get('priceLevel'),
        'website': place.get('websiteUri'),
        'maps_url': place.get('googleMapsUri'),
        'opening_hours_raw': json.dumps(place.get('regularOpeningHours'), ensure_ascii=False) if place.get('regularOpeningHours') else None,
        'current_opening_hours_raw': json.dumps(place.get('currentOpeningHours'), ensure_ascii=False) if place.get('currentOpeningHours') else None,
        'raw': json.dumps(place, ensure_ascii=False)
    }

# Read from the widget cell above.
google_types = list(google_types_widget.value)
extras = [t.strip() for t in google_types_extra.value.split(',') if t.strip()]
# Preserve order, dedupe.
google_types = list(dict.fromkeys(google_types + extras))
max_count = google_max_per_type.value

print(f'Querying {len(google_types)} type(s) at up to {max_count} result(s) each: {google_types}')

google_rows = []

if GOOGLE_API_KEY and google_types:
    for t in google_types:
        print('Querying Google type:', t)
        try:
            result = google_nearby_search(t, max_result_count=max_count)
        except requests.HTTPError as e:
            print('  Request failed:', e)
            continue
        save_json(result, RAW_DIR / f'google_nearby_{t}.json')
        for p in result.get('places', []):
            google_rows.append(flatten_google_place(p, query_type=t))
        time.sleep(0.2)
elif not GOOGLE_API_KEY:
    print('Skipping Google because no API key was provided.')
else:
    print('No Google types selected.')

# Deduplicate by source_id.
google_df = pd.DataFrame(google_rows)
if len(google_df):
    google_df = google_df.drop_duplicates(subset=['source_id'])
    google_df.to_csv(OUT_DIR / 'google_places_flat.csv', index=False)
    save_geojson(google_df.to_dict('records'), OUT_DIR / 'google_places.geojson')

google_df.head()


In [ ]:
if len(google_df):
    display(google_df[['name', 'category_original', 'rating', 'review_count', 'business_status', 'address']].head(10))
    plot_counts(google_df, 'category_original', 'Google Places: top original categories')
    if google_df['rating'].notna().any():
        google_df['rating'].dropna().plot(
            kind='hist', bins=20, figsize=(7, 3.5),
            title='Google Places: rating distribution'
        )
        plt.xlabel('Rating')
        plt.tight_layout()
        plt.show()
    display(exploratory_map(google_df, title='Google Places · click clusters and markers'))
else:
    print('No Google Places results to display.')


## Optional: Google Place Details

Use Place Details when you already have a place ID and want more fields. This can trigger different billing tiers depending on the fields requested.

For class, request a small number of records and a small field mask. You can expand the mask after checking the documentation and understanding billing.


In [ ]:
GOOGLE_DETAILS_FIELD_MASK = ','.join([
    'id',
    'displayName',
    'formattedAddress',
    'location',
    'primaryType',
    'types',
    'businessStatus',
    'rating',
    'userRatingCount',
    'regularOpeningHours',
    'accessibilityOptions',
    'websiteUri',
    'googleMapsUri'
])


def google_place_details(place_id: str) -> Dict:
    if not GOOGLE_API_KEY:
        raise ValueError('No Google API key provided')
    url = GOOGLE_DETAILS_URL.format(place_id=place_id)
    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': GOOGLE_API_KEY,
        'X-Goog-FieldMask': GOOGLE_DETAILS_FIELD_MASK
    }
    r = requests.get(url, headers=headers, timeout=30)
    if r.status_code != 200:
        print(r.status_code, r.text[:1000])
    r.raise_for_status()
    return r.json()

# Optional, limited test on the first 3 Google results.
run_google_details = False
if run_google_details and len(google_df):
    details = []
    for pid in google_df['source_id'].head(3):
        d = google_place_details(pid)
        details.append(d)
        time.sleep(0.2)
    save_json(details, RAW_DIR / 'google_place_details_sample.json')
    details[:1]
else:
    print('Set run_google_details = True to run a small Place Details sample.')

# 03. Foursquare Places API

Foursquare is useful for querying venues with rich metadata and, depending on access and availability, behavioral proxy fields like `popularity` and `hours_popular`.

> **Heads up: the v3 endpoint at `api.foursquare.com/v3/places/search` was deprecated on May 15, 2026.**
> This notebook uses the current "Places API" at `places-api.foursquare.com`, which requires a **Service API Key** from the [Foursquare developer console](https://docs.foursquare.com/developer/docs/manage-service-api-keys), `Authorization: Bearer <key>`, and an `X-Places-Api-Version` header. If you have an old client-style key (long alphanumeric beginning with `fsq3...`), it will not authenticate against the new endpoint — create a Service API Key.

Useful fields to test:

```text
fsq_place_id,name,latitude,longitude,location,categories,distance,closed_bucket,
timezone,hours,hours_popular,rating,stats,popularity,price,tastes,features,
venue_reality_bucket,website,tel
```

Some fields — `hours_popular`, `popularity`, `tastes`, `venue_reality_bucket`, `stats`, `tips`, `photos` — are part of Foursquare's Pro / Premium tiers and may be empty for free-tier keys. Treat missing fields as part of the analysis: which platforms hide their behavioral data behind a paywall, and what does that mean for who can see what?


In [ ]:
# FOURSQUARE_API_KEY is defined in the consolidated API key setup cell above.
# Current Places API endpoint (the legacy v3 host was deprecated May 15, 2026).
FOURSQUARE_SEARCH_URL = 'https://places-api.foursquare.com/places/search'

# Pin a recent API version date. Bump as Foursquare publishes new versions.
FOURSQUARE_API_VERSION = '2025-02-05'

FOURSQUARE_FIELDS = ','.join([
    'fsq_place_id', 'name', 'latitude', 'longitude', 'location',
    'categories', 'distance', 'closed_bucket', 'timezone',
    'hours', 'hours_popular', 'rating', 'stats', 'popularity',
    'price', 'tastes', 'features', 'venue_reality_bucket',
    'website', 'tel',
])
print(FOURSQUARE_FIELDS)


In [ ]:
# --- FOURSQUARE CONTROLS -----------------------------------------------------
# One query per line. Leave a blank line (or write "(broad)") to do a broad
# nearby search with no text query. Hard cap is 50 results per query.

foursquare_queries_widget = widgets.Textarea(
    value='(broad)\ncoffee\nlibrary\npark',
    description='Queries:',
    placeholder='one per line; (broad) = no query',
    layout=widgets.Layout(width='600px', height='110px'),
    style={'description_width': '90px'},
)

foursquare_limit_widget = widgets.IntSlider(
    value=50, min=1, max=50, step=1,
    description='Limit / query:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '110px'},
)
foursquare_limit_help = widgets.HTML(
    "<div style='font-size:11px;color:#666;margin-left:115px;'>"
    "Hard cap is 50 per call (Foursquare). Total calls = number of queries."
    "</div>"
)

display(widgets.VBox([
    widgets.HTML("<b>Foursquare — query controls</b>"),
    foursquare_queries_widget,
    foursquare_limit_widget,
    foursquare_limit_help,
]))


In [ ]:
# FOURSQUARE_API_KEY is defined in the consolidated API key setup cell above.
def foursquare_search(query: Optional[str] = None, sort: str = 'RELEVANCE', limit: int = 50) -> Dict:
    if not FOURSQUARE_API_KEY:
        raise ValueError('No Foursquare API key provided')
    params = {
        'll': f'{CENTER_LAT},{CENTER_LON}',
        'radius': RADIUS_METERS,
        'limit': min(limit, 50),
        'sort': sort,
        'fields': FOURSQUARE_FIELDS,
    }
    if query:
        params['query'] = query
    headers = {
        'Accept': 'application/json',
        'Authorization': f'Bearer {FOURSQUARE_API_KEY}',
        'X-Places-Api-Version': FOURSQUARE_API_VERSION,
    }
    r = requests.get(FOURSQUARE_SEARCH_URL, headers=headers, params=params, timeout=30)
    if r.status_code != 200:
        print(r.status_code, r.text[:1000])
    r.raise_for_status()
    return r.json()


def flatten_foursquare_place(place: Dict, query: Optional[str] = None) -> Dict:
    # Coordinates may come back as flat latitude/longitude or nested under geocodes.main
    # depending on the fields requested. Handle both.
    geocodes = place.get('geocodes', {}) or {}
    main = geocodes.get('main', {}) or {}
    lat = first_nonempty(place.get('latitude'), main.get('latitude'))
    lon = first_nonempty(place.get('longitude'), main.get('longitude'))

    categories = place.get('categories') or []
    category_names = [c.get('name') for c in categories if c.get('name')]

    location = place.get('location', {}) or {}
    address = first_nonempty(
        location.get('formatted_address'),
        ', '.join([str(location.get(k)) for k in ['address', 'locality', 'region', 'postcode'] if location.get(k)])
    )

    # Field name changed from fsq_id (v3) to fsq_place_id (new API). Support both.
    source_id = first_nonempty(place.get('fsq_place_id'), place.get('fsq_id'))

    return {
        'source': 'foursquare',
        'source_id': source_id,
        'name': place.get('name'),
        'category_original': ', '.join(category_names) if category_names else None,
        'query': query,
        'lat': lat,
        'lon': lon,
        'address': address,
        'closed_bucket': place.get('closed_bucket'),
        'rating': place.get('rating'),
        'review_count': safe_get(place, ['stats', 'total_ratings']),
        'price': place.get('price'),
        'popularity': place.get('popularity'),
        'website': place.get('website'),
        'phone': place.get('tel'),
        'hours_raw': json.dumps(place.get('hours'), ensure_ascii=False) if place.get('hours') else None,
        'popular_hours_raw': json.dumps(place.get('hours_popular'), ensure_ascii=False) if place.get('hours_popular') else None,
        'venue_reality_bucket': place.get('venue_reality_bucket'),
        'raw': json.dumps(place, ensure_ascii=False)
    }


def _parse_foursquare_queries(raw_text: str):
    """Parse the textarea into a list of query strings. (broad) and blank lines mean no query."""
    out = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line or line.lower() == '(broad)':
            out.append(None)
        else:
            out.append(line)
    # Dedupe while preserving order.
    seen = set()
    uniq = []
    for q in out:
        key = '' if q is None else q.lower()
        if key not in seen:
            seen.add(key)
            uniq.append(q)
    return uniq


foursquare_queries = _parse_foursquare_queries(foursquare_queries_widget.value)
foursquare_limit = foursquare_limit_widget.value
print(f'Running {len(foursquare_queries)} Foursquare quer(ies) at limit {foursquare_limit} each:')
for q in foursquare_queries:
    print('  ', q if q else '(broad)')

foursquare_rows = []

if FOURSQUARE_API_KEY and foursquare_queries:
    for q in foursquare_queries:
        label = q or 'broad nearby search'
        print('Querying Foursquare:', label)
        try:
            result = foursquare_search(query=q, sort='RELEVANCE', limit=foursquare_limit)
        except requests.HTTPError as e:
            print('  Request failed:', e)
            continue
        save_json(result, RAW_DIR / f'foursquare_search_{q or "nearby"}.json')
        # New API returns results under "results"; older v3 also used "results". Handle either.
        results = result.get('results') or result.get('places') or []
        for p in results:
            foursquare_rows.append(flatten_foursquare_place(p, query=q))
        time.sleep(0.2)
elif not FOURSQUARE_API_KEY:
    print('Skipping Foursquare because no API key was provided.')
else:
    print('No Foursquare queries specified.')

foursquare_df = pd.DataFrame(foursquare_rows)
if len(foursquare_df):
    foursquare_df = foursquare_df.drop_duplicates(subset=['source_id'])
    foursquare_df.to_csv(OUT_DIR / 'foursquare_flat.csv', index=False)
    save_geojson(foursquare_df.to_dict('records'), OUT_DIR / 'foursquare_places.geojson')

foursquare_df.head()


In [ ]:
if len(foursquare_df):
    display(foursquare_df[['name', 'category_original', 'rating', 'popularity', 'closed_bucket', 'address']].head(10))
    plot_counts(foursquare_df, 'category_original', 'Foursquare: top original categories')
    if 'popularity' in foursquare_df.columns and foursquare_df['popularity'].notna().any():
        foursquare_df['popularity'].plot(kind='hist', bins=20, figsize=(7, 4), title='Foursquare popularity distribution')
        plt.xlabel('Popularity score')
        plt.tight_layout()
        plt.show()
    display(exploratory_map(foursquare_df, title='Foursquare · click clusters and markers'))
else:
    print('No Foursquare results to display.')


# 04. OpenStreetMap via Overpass

OpenStreetMap does not have one universal POI field. POI-like features are usually queried through tags such as:

```text
amenity, shop, tourism, leisure, healthcare, office, craft, public_transport, railway
```

This section queries Overpass directly with Python. You can also paste the query into https://overpass-turbo.eu/ and export GeoJSON manually.


In [ ]:
# --- OPENSTREETMAP / OVERPASS CONTROLS ---------------------------------------
# Select which OSM tag keys to fetch. Each key contributes one node + way + relation
# query, so picking too many can produce a very large response on big bboxes.

OSM_KEY_OPTIONS = [
    'amenity', 'shop', 'tourism', 'leisure', 'healthcare',
    'office', 'craft', 'historic', 'sport', 'public_transport',
    'railway', 'building',
]

osm_keys_widget = widgets.SelectMultiple(
    options=OSM_KEY_OPTIONS,
    value=('amenity', 'shop', 'tourism', 'leisure', 'healthcare', 'office', 'craft'),
    description='OSM keys:',
    rows=10,
    layout=widgets.Layout(width='420px'),
    style={'description_width': '90px'},
)

osm_max_widget = widgets.IntSlider(
    value=2000, min=100, max=20000, step=100,
    description='Max kept:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '110px'},
)

osm_no_limit_widget = widgets.Checkbox(
    value=False, description='No limit (keep all)',
    indent=False,
)

def _toggle_osm_max(_change):
    osm_max_widget.disabled = osm_no_limit_widget.value
osm_no_limit_widget.observe(_toggle_osm_max, names='value')

osm_max_help = widgets.HTML(
    "<div style='font-size:11px;color:#666;margin-left:115px;'>"
    "Overpass returns everything in the bbox; this trims the result after fetch. "
    "Check 'No limit' to keep everything."
    "</div>"
)

display(widgets.VBox([
    widgets.HTML("<b>OpenStreetMap — query controls</b>"),
    osm_keys_widget,
    widgets.HBox([osm_max_widget, osm_no_limit_widget]),
    osm_max_help,
]))


In [ ]:
OVERPASS_URL = 'https://overpass-api.de/api/interpreter'

# Read selected keys from the widget above.
osm_keys = list(osm_keys_widget.value)


def build_overpass_query(bbox, keys):
    min_lon, min_lat, max_lon, max_lat = bbox
    bbox_str = f'{min_lat},{min_lon},{max_lat},{max_lon}'
    parts = []
    for key in keys:
        parts.append(f'node["{key}"]({bbox_str});')
        parts.append(f'way["{key}"]({bbox_str});')
        parts.append(f'relation["{key}"]({bbox_str});')
    query = '[out:json][timeout:25];\n(\n  ' + '\n  '.join(parts) + '\n);\nout center tags;'
    return query

osm_query = build_overpass_query(BBOX, osm_keys)
print(f'Querying OSM keys: {osm_keys}')
print(osm_query[:1200])


In [ ]:
# The Overpass operators block the default `python-requests/x.y` user agent and
# will return HTTP 406 Not Acceptable. Always identify yourself with a real UA.
OVERPASS_HEADERS = {
    'User-Agent': 'ARCH6133-POI-tutorial/1.0 (educational; contact: course staff)'
}


def run_overpass(query: str) -> Dict:
    r = requests.post(
        OVERPASS_URL,
        data={'data': query},
        headers=OVERPASS_HEADERS,
        timeout=180,
    )
    if r.status_code != 200:
        print(r.status_code, r.text[:1000])
    r.raise_for_status()
    return r.json()


def flatten_osm_element(el: Dict) -> Dict:
    tags = el.get('tags', {}) or {}
    lat = el.get('lat') or safe_get(el, ['center', 'lat'])
    lon = el.get('lon') or safe_get(el, ['center', 'lon'])
    original_parts = []
    for k in osm_keys:
        if tags.get(k):
            original_parts.append(f'{k}={tags.get(k)}')
    return {
        'source': 'openstreetmap',
        'source_id': f"{el.get('type')}/{el.get('id')}",
        'osm_type': el.get('type'),
        'name': tags.get('name'),
        'category_original': '; '.join(original_parts) if original_parts else None,
        'lat': lat,
        'lon': lon,
        'address': first_nonempty(
            tags.get('addr:full'),
            ' '.join([str(tags.get(k)) for k in ['addr:housenumber', 'addr:street'] if tags.get(k)])
        ),
        'website': first_nonempty(tags.get('website'), tags.get('contact:website')),
        'phone': first_nonempty(tags.get('phone'), tags.get('contact:phone')),
        'opening_hours': tags.get('opening_hours'),
        'raw_tags': json.dumps(tags, ensure_ascii=False),
        'raw': json.dumps(el, ensure_ascii=False)
    }


osm_result = run_overpass(osm_query)
save_json(osm_result, RAW_DIR / 'osm_overpass_raw.json')
osm_rows = [flatten_osm_element(el) for el in osm_result.get('elements', [])]

osm_df = pd.DataFrame(osm_rows)
if len(osm_df):
    osm_df = osm_df.drop_duplicates(subset=['source_id'])
    fetched = len(osm_df)
    if not osm_no_limit_widget.value and fetched > osm_max_widget.value:
        # Prefer named features first when trimming.
        osm_df = osm_df.sort_values('name', na_position='last').head(osm_max_widget.value).reset_index(drop=True)
        print(f'Fetched {fetched} OSM elements; trimmed to {len(osm_df)} (named-first).')
    else:
        print(f'Fetched {fetched} OSM elements; kept all.')
    osm_df.to_csv(OUT_DIR / 'osm_flat.csv', index=False)
    save_geojson(osm_df.to_dict('records'), OUT_DIR / 'osm_places.geojson')

osm_df.head()


In [ ]:
if len(osm_df):
    display(osm_df[['name', 'category_original', 'opening_hours', 'website', 'address']].head(10))
    plot_counts(osm_df, 'category_original', 'OpenStreetMap: top original tag combinations', top_n=20)
    display(exploratory_map(osm_df, title='OpenStreetMap · click clusters and markers'))
else:
    print('No OSM results to display.')


# 05. Overture Maps Places

Overture Maps distributes open places data as cloud-hosted GeoParquet. This is different from calling a search API: you can query or download the data you need and build your own platform logic from it.

The current Places Guide describes the `place` feature type as point representations of real-world entities. Key fields include `id`, `names`, `categories`, `confidence`, `sources`, `websites`, `socials`, `emails`, `phones`, `addresses`, `operating_status`, and geometry.

There are two ways to work with Overture in class. The notebook uses (1) by default; (2) is provided as a reference for students who want to learn the SQL/infrastructure side.

1. **Default — Overture Python client / CLI.** Downloads a clipped GeoJSON for your bbox. Works in Colab once `overturemaps` is installed.
2. **Advanced — DuckDB against the public GeoParquet.** Lets you write SQL filters across the raw release without downloading the whole file. Useful for teaching data infrastructure.

Overture publishes a new release roughly monthly. The notebook pins a known-good release below; bump it when you start a new project. You can list current releases at the [Overture releases page](https://docs.overturemaps.org/release/latest/).


In [ ]:
# --- OVERTURE CONTROLS -------------------------------------------------------
# The Overture CLI downloads everything in your bbox; the filters below are
# applied after the download in the loader cell.

overture_max_widget = widgets.IntSlider(
    value=5000, min=100, max=50000, step=100,
    description='Max kept:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '110px'},
)

overture_no_limit_widget = widgets.Checkbox(
    value=False, description='No limit (keep all)',
    indent=False,
)

def _toggle_overture_max(_change):
    overture_max_widget.disabled = overture_no_limit_widget.value
overture_no_limit_widget.observe(_toggle_overture_max, names='value')

overture_min_conf_widget = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.05,
    description='Min confidence:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '120px'},
    readout_format='.2f',
)
overture_help = widgets.HTML(
    "<div style='font-size:11px;color:#666;margin-left:125px;'>"
    "Overture publishes a per-record confidence score (0-1). Higher = the conflated source agreement is stronger."
    "</div>"
)

display(widgets.VBox([
    widgets.HTML("<b>Overture — post-download filters</b>"),
    widgets.HBox([overture_max_widget, overture_no_limit_widget]),
    overture_min_conf_widget,
    overture_help,
]))


In [ ]:
# Option A: Overture Python client / command-line tool.
# In Colab you may need: !pip -q install overturemaps
#
# This runs by default. Set run_overture_cli = False below to skip.

import subprocess
import shutil

# Pin a known-good monthly release. Bump this when you start a new project.
OVERTURE_RELEASE = '2026-06-17.0'

bbox_str = ','.join(str(c) for c in BBOX)
overture_geojson_path = OUT_DIR / 'overture_places.geojson'

run_overture_cli = True

if run_overture_cli:
    if shutil.which('overturemaps') is None:
        print('overturemaps CLI not found. In Colab run:  !pip -q install overturemaps')
    else:
        cmd = [
            'overturemaps', 'download',
            '--bbox', bbox_str,
            '-f', 'geojson',
            '--type', 'place',
            '-o', str(overture_geojson_path),
        ]
        print('Running:', ' '.join(cmd))
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.stdout:
            print(result.stdout.strip())
        if result.returncode != 0:
            print('STDERR:', result.stderr.strip()[:2000])
        else:
            size_kb = overture_geojson_path.stat().st_size / 1024
            print(f'Saved {overture_geojson_path}  ({size_kb:.1f} KB)')
else:
    print('Set run_overture_cli = True to download Overture Places for the bbox.')


In [ ]:
# Option B (advanced, optional): DuckDB SQL against Overture's public GeoParquet.
# In Colab you may need: !pip -q install duckdb
#
# This cell is OFF by default. Flip run_overture_duckdb = True to execute.
# It overwrites the same overture_places.geojson that Option A produces.

run_overture_duckdb = False

DUCKDB_SQL = f"""
INSTALL spatial; LOAD spatial;
INSTALL httpfs; LOAD httpfs;
SET s3_region='us-west-2';

COPY (
  SELECT
    id,
    version,
    names.primary AS name,
    categories.primary AS category,
    basic_category,
    confidence,
    operating_status,
    CAST(sources AS JSON) AS sources,
    CAST(websites AS JSON) AS websites,
    CAST(addresses AS JSON) AS addresses,
    geometry
  FROM read_parquet(
    's3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/theme=places/type=place/*',
    filename=true,
    hive_partitioning=1
  )
  WHERE
    bbox.xmin BETWEEN {BBOX[0]} AND {BBOX[2]}
    AND bbox.ymin BETWEEN {BBOX[1]} AND {BBOX[3]}
    AND confidence > 0.60
) TO '{overture_geojson_path}' WITH (FORMAT GDAL, DRIVER 'GeoJSON', SRS 'EPSG:4326');
"""

if run_overture_duckdb:
    import duckdb
    con = duckdb.connect()
    con.execute(DUCKDB_SQL)
    con.close()
    print('DuckDB wrote:', overture_geojson_path)
else:
    print('SQL prepared (not executed). Set run_overture_duckdb = True to run it.')
    print(DUCKDB_SQL[:1200])


In [ ]:
def load_overture_geojson(path: Path) -> pd.DataFrame:
    if not path.exists():
        print('No Overture file found yet:', path)
        return pd.DataFrame()
    gj = load_json(path)
    rows = []
    for f in gj.get('features', []):
        props = f.get('properties', {}) or {}
        coords = safe_get(f, ['geometry', 'coordinates'], [None, None])
        addresses = props.get('addresses') or []
        primary_address = addresses[0] if addresses else {}
        websites = props.get('websites') or []
        rows.append({
            'source': 'overture',
            # `id` is at the Feature root in CLI output and inside properties in DuckDB output.
            'source_id': f.get('id') or props.get('id'),
            'name': props.get('name') or safe_get(props, ['names', 'primary']),
            'category_original': first_nonempty(
                props.get('category'),
                safe_get(props, ['categories', 'primary']),
                props.get('basic_category'),
            ),
            'lat': coords[1] if coords else None,
            'lon': coords[0] if coords else None,
            'confidence': props.get('confidence'),
            'operating_status': props.get('operating_status'),
            'address': first_nonempty(
                props.get('address'),
                primary_address.get('freeform') if isinstance(primary_address, dict) else None,
            ),
            'website': websites[0] if websites else None,
            'sources_raw': json.dumps(props.get('sources'), ensure_ascii=False) if props.get('sources') else None,
            'raw': json.dumps(props, ensure_ascii=False)
        })
    return pd.DataFrame(rows)

overture_df = load_overture_geojson(overture_geojson_path)
if len(overture_df):
    # Clip to the bbox we asked for (CLI download includes a buffered area).
    overture_df = overture_df[
        overture_df['lon'].between(BBOX[0], BBOX[2])
        & overture_df['lat'].between(BBOX[1], BBOX[3])
    ].reset_index(drop=True)
    fetched = len(overture_df)

    # Apply widget filters.
    min_conf = overture_min_conf_widget.value
    if min_conf > 0 and 'confidence' in overture_df.columns:
        overture_df = overture_df[overture_df['confidence'].fillna(0) >= min_conf].reset_index(drop=True)

    if not overture_no_limit_widget.value and len(overture_df) > overture_max_widget.value:
        # Highest-confidence first when trimming.
        overture_df = (
            overture_df.sort_values('confidence', ascending=False, na_position='last')
            .head(overture_max_widget.value)
            .reset_index(drop=True)
        )
        print(f'Loaded {fetched} Overture rows in bbox; trimmed to {len(overture_df)} (highest confidence first, min_conf={min_conf}).')
    else:
        print(f'Loaded {fetched} Overture rows in bbox; kept {len(overture_df)} after filters (min_conf={min_conf}).')

    overture_df.to_csv(OUT_DIR / 'overture_flat.csv', index=False)

overture_df.head()


In [ ]:
if len(overture_df):
    display(overture_df[['name', 'category_original', 'confidence', 'operating_status']].head(10))
    plot_counts(overture_df, 'category_original', 'Overture: top original categories', top_n=20)
    if 'confidence' in overture_df.columns and overture_df['confidence'].notna().any():
        overture_df['confidence'].plot(kind='hist', bins=20, figsize=(7, 4), title='Overture confidence distribution')
        plt.xlabel('Confidence')
        plt.tight_layout()
        plt.show()
    display(exploratory_map(overture_df, title='Overture · click clusters and markers'))
else:
    print('No Overture results to display.')


# 06. Normalize categories across sources

Normalization is an interpretive act. Preserve each source's original category, then add a standardized field that helps you compare.

Edit the function below as your categories become more specific.


In [ ]:
def standardize_category(cat: Any) -> str:
    if pd.isna(cat):
        return 'unknown'
    c = str(cat).lower()
    food_terms = ['restaurant', 'cafe', 'coffee', 'bar', 'bakery', 'food', 'pizza', 'deli', 'drink']
    retail_terms = ['shop', 'store', 'retail', 'supermarket', 'grocery', 'market', 'pharmacy']
    culture_terms = ['museum', 'gallery', 'theatre', 'theater', 'library', 'cultural', 'arts']
    education_terms = ['school', 'college', 'university', 'education', 'kindergarten']
    health_terms = ['hospital', 'clinic', 'doctor', 'dentist', 'healthcare', 'pharmacy']
    religion_terms = ['church', 'mosque', 'synagogue', 'temple', 'religion', 'place_of_worship']
    transit_terms = ['transit', 'subway', 'bus', 'railway', 'station']
    park_terms = ['park', 'garden', 'playground', 'open_space', 'recreation']
    service_terms = ['bank', 'atm', 'laundry', 'post_office', 'service', 'office']
    government_terms = ['government', 'courthouse', 'police', 'fire_station', 'townhall']

    tests = [
        ('food_drink', food_terms),
        ('retail', retail_terms),
        ('culture', culture_terms),
        ('education', education_terms),
        ('health', health_terms),
        ('religion', religion_terms),
        ('transit', transit_terms),
        ('park_open_space', park_terms),
        ('service', service_terms),
        ('government', government_terms),
    ]
    for label, terms in tests:
        if any(t in c for t in terms):
            return label
    return 'other'

source_dfs = []
for df in [google_df, foursquare_df, osm_df, overture_df]:
    if len(df):
        source_dfs.append(df.copy())

combined_df = pd.concat(source_dfs, ignore_index=True, sort=False) if source_dfs else pd.DataFrame()
if len(combined_df):
    combined_df['category_standardized'] = combined_df['category_original'].apply(standardize_category)
    keep_cols = [
        'source', 'source_id', 'name', 'category_original', 'category_standardized',
        'lat', 'lon', 'address', 'website', 'phone', 'opening_hours', 'opening_hours_raw',
        'rating', 'review_count', 'price', 'popularity', 'popular_hours_raw',
        'confidence', 'business_status', 'operating_status', 'closed_bucket', 'notes'
    ]
    for col in keep_cols:
        if col not in combined_df.columns:
            combined_df[col] = None
    combined_export = combined_df[keep_cols].copy()
    combined_export.to_csv(OUT_DIR / 'combined_poi_normalized.csv', index=False)
    save_geojson(combined_export.to_dict('records'), OUT_DIR / 'combined_poi_normalized.geojson')

combined_df.head()

In [ ]:
if len(combined_df):
    display(combined_df[['source', 'name', 'category_original', 'category_standardized']].head(20))
    plot_counts(combined_df, 'source', 'Records by source')
    plot_counts(combined_df, 'category_standardized', 'Records by standardized category')
    # Color by source so the four platforms are visually distinguishable; toggle layers off/on
    # to compare what each one sees.
    display(exploratory_map(combined_df, color_by='source', title='All sources · color by source'))


# 07. Compare metadata availability

Instead of asking only "which source has more points?" ask "which source has which kinds of metadata?"

This section creates a simple metadata availability table. A value counts as available if it is not null and not an empty string.


In [ ]:
metadata_fields = [
    'address', 'website', 'phone', 'opening_hours', 'opening_hours_raw',
    'rating', 'review_count', 'price', 'popularity', 'popular_hours_raw',
    'confidence', 'business_status', 'operating_status', 'closed_bucket'
]

def availability_by_source(df: pd.DataFrame, fields: List[str]) -> pd.DataFrame:
    rows = []
    for source, sub in df.groupby('source'):
        total = len(sub)
        row = {'source': source, 'record_count': total}
        for field in fields:
            if field in sub.columns:
                available = sub[field].notna() & (sub[field].astype(str).str.len() > 0) & (sub[field].astype(str) != 'None')
                row[field] = int(available.sum())
                row[f'{field}_pct'] = round(100 * available.sum() / total, 1) if total else 0
            else:
                row[field] = 0
                row[f'{field}_pct'] = 0
        rows.append(row)
    return pd.DataFrame(rows)

if len(combined_df):
    avail = availability_by_source(combined_df, metadata_fields)
    display(avail[['source', 'record_count'] + [f'{f}_pct' for f in metadata_fields if f in combined_df.columns]].round(1))
    avail.to_csv(OUT_DIR / 'metadata_availability_by_source.csv', index=False)
else:
    print('No combined data yet.')

In [ ]:
if len(combined_df):
    # Build a long-form table for plotting.
    pct_cols = [c for c in avail.columns if c.endswith('_pct')]
    long_avail = avail.melt(id_vars='source', value_vars=pct_cols, var_name='field', value_name='percent_available')
    long_avail['field'] = long_avail['field'].str.replace('_pct', '', regex=False)
    display(long_avail.head())

    # Simple chart: one source at a time. Change selected_source.
    selected_source = long_avail['source'].iloc[0]
    plot_df = long_avail[long_avail['source'] == selected_source].sort_values('percent_available')
    ax = plot_df.plot(x='field', y='percent_available', kind='barh', legend=False, figsize=(8, 5), title=f'Metadata availability: {selected_source}')
    ax.set_xlabel('Percent of records with field available')
    ax.set_ylabel('Field')
    plt.tight_layout()
    plt.show()

# 08. Source comparison prompts

Write directly in this notebook or in a separate document.

## Prompt 1: What appears?

Which places appear across multiple platforms? Which places appear in only one? What might explain the difference?

## Prompt 2: What categories dominate?

Which categories are most common in each source? What does that suggest about the platform's model of place?

## Prompt 3: What metadata changes your understanding?

Does ratings data, popular hours, confidence, operating status, opening hours, or source attribution change how you understand this area?

## Prompt 4: What is missing?

List at least five places, uses, practices, atmospheres, histories, or social relations that matter in the study area but do not appear in the data.

## Prompt 5: What would you need to make yourself?

Would you need field observation, sensing, interviews, OCR, imagery, archival work, participatory mapping, or semantic annotation to represent what is missing?


In [ ]:
reflection_template = f"""
# POI Platform Audit Reflection

Study area: {STUDY_AREA_NAME}
Bounding box: {BBOX}

## 1. What does each source think a place is?

Google Places:

Foursquare:

OpenStreetMap:

Overture Maps:

## 2. What appears and what is missing?

Places that appeared across multiple systems:

Places that appeared in only one system:

Places that were missing from all systems:

## 3. Metadata critique

Fields that changed how I understood the area:

Fields I wanted but could not access:

Fields that felt ethically or politically sensitive:

## 4. Prototype translation

My final platform might use POI data to:

POI data would be insufficient because:

Fields I would need to create myself:

Potential risks of making these places visible:
"""

reflection_path = OUT_DIR / 'poi_platform_audit_reflection_template.md'
reflection_path.write_text(reflection_template, encoding='utf-8')
print(reflection_template)
print('Saved:', reflection_path)

# 09. Export checklist

Before submitting, make sure you have:

- Raw files from each source you queried in `poi_data/raw/`.
- Flattened source CSVs in `poi_data/output/`.
- `combined_poi_normalized.csv`.
- `combined_poi_normalized.geojson`.
- `metadata_availability_by_source.csv`.
- A short reflection using the template above.
- One screenshot or exported map showing your study area and points.

## Final critical reminder

Do not write: "Google shows the real places" or "OSM is the accurate version."

Instead write: "This source makes some places visible through a particular schema, infrastructure, and user model."
